In [ ]:
import pandas as pd

In [ ]:
# Load the datasets
vneconomy_df = pd.read_csv('../../data/raw/scraped/vneconomy.csv')
baodautu_df = pd.read_csv('../../data/raw/scraped/baodautu.csv')

# Add a source column to each DataFrame
vneconomy_df['source'] = 'vneconomy'
baodautu_df['source'] = 'baodautu'

In [ ]:
vne_time = vneconomy_df.sample(1).iloc[0]['time']
bdt_time = baodautu_df.sample(1).iloc[0]['time']

print(vne_time)
print(bdt_time)

vneconomy_df['time'] = pd.to_datetime(vneconomy_df['time']).dt.date
baodautu_df['time'] = pd.to_datetime(baodautu_df['time'], format='%Y-%m-%d').dt.date

In [ ]:
# Concatenate the DataFrames
df = pd.concat([vneconomy_df, baodautu_df], ignore_index=True)

print(f"Shape of vneconomy_df: {vneconomy_df.shape}")
print(f"Shape of baodautu_df: {baodautu_df.shape}")
print(f"Shape of merged_df: {df.shape}")

In [ ]:
df.info()

In [ ]:
from transformers import AutoTokenizer

model_name = "vinai/bartpho-syllable"
tokenizer = AutoTokenizer.from_pretrained(model_name)

df['tokenized_title'] = df['title'].apply(lambda x: tokenizer.encode(str(x), truncation=False))
df['tokenized_content'] = df['content'].apply(lambda x: tokenizer.encode(str(x), truncation=False))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_token_distribution(column, color, bins=50):
    """
    Plots histogram and KDE of a token count column in separate figures,
    with a describe() summary table alongside the histogram.
    
    Args:
        df (pd.DataFrame): The dataframe containing the token column.
        column (str): The name of the column (e.g., 'content_token_counts').
        bins (int): Number of bins for the histogram (default=50).
    """
    token_counts = df[column].apply(len)
    stats = token_counts.describe().round(2)

    # Histogram + Stats Table
    fig, ax = plt.subplots(1, 2, figsize=(10, 5), gridspec_kw={'width_ratios': [3, 1]})
    
    # Histogram
    ax[0].hist(token_counts, bins=bins, color=color, edgecolor='black')
    ax[0].set_title(f"{column} Histogram")
    ax[0].set_xlabel("Number of Tokens")
    ax[0].set_ylabel("Number of Articles")
    ax[0].grid(True)

    # Describe table on the side
    cell_text = [[f"{val:.2f}"] for val in stats]
    row_labels = stats.index.tolist()
    table = ax[1].table(cellText=cell_text,
                        rowLabels=row_labels,
                        colLabels=["Value"],
                        loc='center',
                        cellLoc='center')
    table.scale(1, 1.5)
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    ax[1].axis('off')
    ax[1].set_title("Descriptive Stats")

    plt.tight_layout()
    plt.show()
    
# plot_token_distribution("tokenized_title", color='coral')
# plot_token_distribution("tokenized_content", color='skyblue')

In [ ]:
df['content_token_counts'] = df['tokenized_content'].apply(len)
df = df[df['content_token_counts'].between(128, 2000)]
print(df.shape)

In [ ]:
# plot_token_distribution("tokenized_title", color='coral')
# plot_token_distribution("tokenized_content", color='skyblue')

In [ ]:
df.head()

In [ ]:
categories = df['category'].value_counts().index.to_numpy()
category_counts = df['category'].value_counts()
df[df['category'] == ""]

for cat in categories:
    sample = df[df['category'] == cat].sample(1).iloc[0]
    print(f"[{category_counts[cat]:<5}] {cat}: {sample['url']}")

In [ ]:
import pandas as pd

# df = pd.read_csv("../../data/scraped/merged_data.csv")

keep_categories = [
    "Doanh nghiệp",
    "Đầu tư",
    "Chứng khoán",
    "Bất động sản",
    # "Thế giới",
    "Tài chính",
    # "Tài chính - Chứng khoán",
    # "Dự án - quy hoạch",
    "Kinh tế số",
    "Thị trường",
    "Chuyển động thị trường",
    "Thông tin doanh nghiệp"
]

category_mapping = {
    "Chuyển động thị trường" : "Thị trường",
}
df['category'] = df['category'].replace(category_mapping)
df = df[df['category'].isin(keep_categories)]
df.shape

In [ ]:
import ast

def clean_and_join_tags(tag_string):
    if pd.isna(tag_string):
        return tag_string

    # Nếu là list string thì parse, nếu không thì giữ nguyên
    try:
        parsed = ast.literal_eval(tag_string)
        if isinstance(parsed, list):
            tags = parsed
        else:
            tags = tag_string.split(',')
    except (ValueError, SyntaxError):
        tags = tag_string.split(',')

    # Loại bỏ "vneconomy" và strip khoảng trắng
    tags = [tag.strip() for tag in tags if tag.strip().lower() != "vneconomy"]

    return ", ".join(tags) if tags else None

df['tags'] = df['tags'].apply(clean_and_join_tags)


keep_cols = [
    "url",
    "title",
    "time",
    "category",
    "content",
    "tags",
    "content_token_counts"
]
df = df[keep_cols]

basepath = "../data_collection/datasets/"
# df.to_csv(basepath + "full_data.csv", index=False)

In [ ]:
counts = df['category'].value_counts()

counts.plot(kind='pie', autopct='%1.1f%%')
plt.ylabel('')
plt.title('Category Distribution')
plt.show()

In [ ]:
df.head()

In [ ]:
df.shape